In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sympy import symbols,solve

In [3]:
# ICA
# signals mixed. x = A.s
# centering (x_i - mean) > covariance matrix > Eigen values, Eigen vectors
#  > whitening M = Eigen_vector/sqrt( diagnol (Eigen values) ) > Xw = np.dot(M,Xc)
# weights update > w_new E[tanh(u)*Xw] - E[1-tanh(u)**2]*w > u = w.T * Xw, where w is weights initialized (normalized)
# mormalize: w_new / np.linalg.norm(w_new)
x1 = np.array([2,4,6,8])
x2 = np.array([7,5,1,3])
x = np.array([x1,x2])
print("x: ")
print(x)
A = np.array([[5,2],[2,1]])
X = np.dot(A,x)
# print("mixed: ",X)
# mixed:  [[24 30 32 46]
#  [11 13 13 19]]
# centring
xcm1 = (24 + 30 + 32 + 46)/4
xcm2 = (11 + 13 + 13 + 19)/4
# print("mean of mixed 1: ",xcm1,"mean of mixed 2: ",xcm2)
xcc1 = (24-33, 30-33, 32-33, 46-33)
xcc2 = (11-14, 13-14, 13-14, 19-14)
# print("centered mixed 1: ",xcc1,"centered mixed 2: ",xcc2)
xv1 = (1/3)*np.dot(xcc1,xcc1)
xv2 = (1/3)*np.dot(xcc2,xcc2)
# print("variance of mixed 1: ",xv1,"variance of mixed 2: ",xv2)
xcv1 = (1/3)*np.dot(xcc1,xcc2)
xcv2 = (1/3)*np.dot(xcc2,xcc1)
# print("covariance of mixed 1: ",xcv1,"covariance of mixed 2: ",xcv2)
cov_mat = np.array([[xv1,xcv1],[xcv2,xv2]])
# print("covariance matrix: ",cov_mat)

e_val,e_vec = np.linalg.eig(cov_mat) # eigen
# print("eigen values: ",e_val)
# print("eigen vectors: ",e_vec)
e_val_M = 1/np.sqrt(e_val)
# print("eigen values M: ",e_val_M)
D = np.array([
    [e_val_M[0], 0],
    [0, e_val_M[1]]
    ])
# print("diagonal matrix: ",D)
M = np.dot(D, e_vec.T)
# print("Whitening the data: ",M)
Xc = np.vstack((xcc1,xcc2))
# print("M shape", M.shape, "centered data shape", Xc.shape)

# multiplying M with centered data
Xw = np.dot(M,Xc)
# print("weighted data: ",Xw) # (2, 4)
# [-0.95535221 -0.31845074 -0.12945494  1.40325789]
# [ 0.76635642  0.25545214 -1.46625649  0.44444793]
w = Xw[:,3] / np.linalg.norm(Xw[:,3])
# print("Initial weight vector:", w)
# [0.95332601 0.30194291]
for i in range(5000):
    w_old = w
    u = np.dot(w, Xw)
    g_u = np.tanh(u)
    g_prime = 1 - g_u**2
    term1 = np.mean(Xw * g_u, axis=1)
    term2 = np.mean(g_prime) * w_old
    w_new = term1 - term2 # Update Rule
    w_new = w_new / np.linalg.norm(w_new) # Normalize
    w = w_new
    # print(f"Iteration {i+1}: w = {w}")
    if np.abs(np.dot(w, w_old)) > 0.9999:
      # print(f"Converged at iteration {i}")
      print("Final weight vector:", w)
      break

x: 
[[2 4 6 8]
 [7 5 1 3]]
Final weight vector: [0.97925158 0.20264832]
